# 6.12 · t-SNE / t-Distributed Stochastic Neighbor Embedding

> **课程定位 / Where this fits**
> PCA/核PCA(6.8/6.9)是降维, 但 PCA 是线性、保全局。t-SNE 是**非线性、保局部邻域**的**可视化**专用降维: 把高维数据(如 784 维 MNIST)摊成 2D, 让同类点聚成肉眼可见的簇。它几乎是高维数据探索的标准工具——但有一堆**易被误读**的坑(簇大小/簇间距离不可信)。
> t-SNE is a nonlinear, neighbour-preserving visualisation technique that lays high-dim data flat into 2D — great for seeing clusters, but with pitfalls (cluster sizes/distances are not meaningful).

> 💡 **面试相关 / Interview-relevant**
> - "t-SNE 的原理(概率相似度 + KL 散度)" ★★★★★
> - "为什么低维用 t 分布(重尾)解决拥挤问题" ★★★★★
> - "perplexity 是什么 / 怎么影响结果" ★★★★★
> - "t-SNE 结果的哪些方面不可信" ★★★★★（簇大小/距离/全局结构）
> - "t-SNE 为什么不能 transform 新数据" ★★★★
> - "t-SNE vs PCA vs UMAP" ★★★★

---

## 学习目标 / Learning Objectives
1. 高维/低维相似度(高斯 vs t 分布)+ KL 目标。
2. 拥挤问题与 t 分布重尾的作用。
3. perplexity 的影响。
4. 正确解读 + 常见误区。
5. t-SNE 的局限(不可 transform、慢、随机)。

## 目录 / TOC
1. [原理: 相似度 + KL ⭐](#1)
2. [🔢 数据: Digits + t-SNE ⭐](#2)
3. [perplexity 的影响 ⭐](#3)
4. [误区: 什么不可信 ⭐](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 原理: 相似度 + KL ⭐ / Similarities & KL Divergence

t-SNE 把"保持邻居关系"形式化成概率匹配:
1. **高维相似度** $p_{ij}$: 用高斯核把点对距离转成"$i$ 选 $j$ 当邻居"的概率——近的点 $p$ 大。每个点的高斯带宽由 **perplexity** 决定(相当于"有效邻居数")。
2. **低维相似度** $q_{ij}$: 在 2D 嵌入里用 **t 分布(自由度1, 即柯西)** 算相似度。
3. **目标**: 最小化两个分布的 **KL 散度** $\mathrm{KL}(P\|Q)=\sum_{ij}p_{ij}\log\frac{p_{ij}}{q_{ij}}$, 用梯度下降挪动低维点。

**为何低维用 t 分布(重尾)—— 拥挤问题(crowding)**: 高维空间"体积"远大于 2D, 把所有邻居塞进 2D 会过度拥挤。t 分布的**重尾**让中等距离的点对在低维**多留点空间**(允许稍微推远), 缓解拥挤, 让簇分得开。KL 的不对称性则让 t-SNE **重点保住近邻**(把近的放近), 而不太管远的。


<a id="2"></a>
## 2. 数据: Digits + t-SNE ⭐ / Digits & t-SNE

**Digits**(64 维手写数字, MNIST 迷你版)。先用 PCA 看线性投影, 再用 t-SNE, 对比簇的清晰度。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
sns.set_theme(style="whitegrid")

digits = load_digits()
X, y = digits.data, digits.target
print(f"Digits: {X.shape}, 10 类 0-9")

# 常规做法: 先 PCA 预降到 ~30 维(降噪+加速), 再 t-SNE / PCA pre-reduction then t-SNE
X30 = PCA(n_components=30, random_state=0).fit_transform(X)
t = time.perf_counter()
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(X30)
print(f"t-SNE 用时 {time.perf_counter()-t:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
p2 = PCA(2, random_state=0).fit_transform(X)
sc0 = axes[0].scatter(p2[:,0], p2[:,1], c=y, cmap="tab10", s=10)
axes[0].set_title("PCA 2D: 数字簇大量重叠(线性投影看不清)")
sc1 = axes[1].scatter(Z[:,0], Z[:,1], c=y, cmap="tab10", s=10)
axes[1].set_title("t-SNE 2D: 10 个数字簇清晰分离")
plt.colorbar(sc1, ax=axes[1], label="digit"); plt.tight_layout(); plt.show()
print("t-SNE 把同类数字聚成清晰簇 → 高维数据探索/可视化标准工具")


<a id="3"></a>
## 3. perplexity 的影响 ⭐ / Effect of Perplexity

**perplexity** ≈ 每个点考虑的"有效邻居数"(常 5–50)。太小→只看极近邻, 结果碎、噪声大; 太大→邻域过宽, 簇糊在一起。它是 t-SNE 最重要的旋钮, **必须多试几个**。


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, perp in zip(axes, [5, 30, 50, 100]):
    Zp = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(X30)
    ax.scatter(Zp[:,0], Zp[:,1], c=y, cmap="tab10", s=6)
    ax.set_title(f"perplexity={perp}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("perplexity: 太小(5)碎裂噪声大; 适中(30-50)清晰; 太大(100)开始糊在一起")
plt.tight_layout(); plt.show()
print("perplexity 是核心超参(有效邻居数, 常 5-50); 不同值结果差异大, 应多试")


<a id="4"></a>
## 4. 误区: 什么不可信 ⭐ / What NOT to Trust

t-SNE 极易被误读(面试高频)。**不要**从 t-SNE 图读出这些结论:
- ❌ **簇的大小**: t-SNE 会把稠密簇放大、稀疏簇缩小, 簇面积**无意义**。
- ❌ **簇间距离**: 两簇离得远≠真的更不相似; **全局距离不保真**(它只保局部邻域)。
- ❌ **空白间隙**: 间隙宽窄不代表什么。
- ⚠️ **随机性**: 每次运行(不同 seed)布局不同; 形状会变。
- ⚠️ **不能 transform 新点**: t-SNE 没有可复用的映射函数(每次都重算整批)→ 不能用于生产推理 pipeline。

✅ **能信的**: 哪些点彼此是近邻(局部结构)、有没有分成几团。它是**探索/可视化**工具, 不是建模/降维特征工具。


In [ ]:
# 演示: 不同 seed → 不同布局(但簇结构一致) / different seeds, different layouts
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, seed in zip(axes, [0, 1, 2]):
    Zs = TSNE(2, perplexity=30, init="random", random_state=seed).fit_transform(X30)
    ax.scatter(Zs[:,0], Zs[:,1], c=y, cmap="tab10", s=6)
    ax.set_title(f"random_state={seed}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("不同随机种子: 簇的相对位置/朝向都变 → 别解读簇间距离与全局布局")
plt.tight_layout(); plt.show()
print("簇内成员稳定, 但簇的相对位置/旋转随机变 → 全局几何不可信")


<a id="5"></a>
## 5. 小结 / Summary

```
t-SNE: 高维相似度 p_ij(高斯, 由perplexity定带宽) ↔ 低维 q_ij(t分布) 匹配, 最小化 KL(P‖Q)
低维用 t 分布(重尾)→ 解决拥挤问题, 给中距离点对留空间, 簇分得开
perplexity(有效邻居数, 5-50)是核心超参, 多试几个
不可信: 簇大小/簇间距离/全局结构/间隙; 随机(每次不同); 不能 transform 新点
能信: 局部邻域 + 是否分团; 是探索/可视化工具, 非降维特征工具
常规: 先 PCA 预降到 ~30-50 维再 t-SNE(降噪+加速)
```

### 💡 面试速查
1. **原理**: 高维高斯相似度 vs 低维 t 分布, 最小化 KL 散度
2. **t 分布重尾解决拥挤问题**(高维体积 >> 2D)
3. **perplexity** = 有效邻居数, 最关键超参
4. **不可信**: 簇大小/簇间距离/全局几何; **不能 transform** 新点(无映射函数)
5. **vs PCA**: 非线性保局部 vs 线性保全局方差; t-SNE 慢、仅可视化

### 下一节
**6.13 UMAP**——和 t-SNE 一样保局部、可视化强, 但**更快、能 transform 新数据、更好地保留一些全局结构**, 已逐渐成为新首选。
